In [ ]:
import json
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from mpl_toolkits.mplot3d import Axes3D

# ============================================================
# Combined Reservoir Comparison Notebook
# Imports the existing Gaussian workflow from gaussian_reservoir_implementation.ipynb
# and the connectome workflow from reservoir.py (the same workflow used in main.ipynb).
# ============================================================

NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / "reservoir.py").exists():
    candidate = Path("4. Reference")
    if (candidate / "reservoir.py").exists():
        NOTEBOOK_DIR = candidate.resolve()

if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from reservoir import run_reservoir_workflow


def load_gaussian_workflow(notebook_path):
    """Load the Gaussian reservoir functions from the existing notebook."""
    nb = json.loads(Path(notebook_path).read_text())
    src = "".join(nb["cells"][0]["source"])

    filtered_lines = []
    for line in src.splitlines():
        if "matplotlib" in line or "plotly" in line or "Axes3D" in line:
            continue
        filtered_lines.append(line)

    namespace = {"np": np, "time": time}
    exec("\n".join(filtered_lines), namespace)
    return namespace["run_gaussian_reservoir_workflow"]


run_gaussian_reservoir_workflow = load_gaussian_workflow(
    NOTEBOOK_DIR / "gaussian_reservoir_implementation.ipynb"
)

WORKFLOW_KEYS = [
    "washout_steps",
    "train_steps",
    "n_steps",
    "W",
    "W_in",
    "W_out",
    "r",
    "u_tr",
    "u_mean",
    "u_std",
    "lorenz_last_state",
    "R_pred",
    "Y_pred_norm",
    "Y_pred",
    "Y_true",
    "lorenz_states",
    "total_runtime",
    "alpha",
]


def workflow_result_to_dict(result):
    data = dict(zip(WORKFLOW_KEYS, result))
    data["N"] = data["W"].shape[0]
    return data


def run_both_reservoirs(common_params, human_params, gaussian_params):
    print("=" * 60)
    print("RUNNING HUMAN CONNECTOME RESERVOIR")
    print("=" * 60)
    human_result = workflow_result_to_dict(
        run_reservoir_workflow(
            file_path=human_params["file_path"],
            J=common_params["J"],
            lambda_reg=common_params["lambda_reg"],
            t_thermalization=common_params["t_thermalization"],
            t_training=common_params["t_training"],
            t_prediction=common_params["t_prediction"],
            sigma=common_params["sigma"],
            rho=common_params["rho"],
            beta=common_params["beta"],
            dt=common_params["dt"],
            input_scale=common_params["input_scale"],
            seed=common_params["seed"],
            alpha=human_params["alpha"],
        )
    )

    print("\n" + "=" * 60)
    print("RUNNING GAUSSIAN RESERVOIR")
    print("=" * 60)
    gaussian_result = workflow_result_to_dict(
        run_gaussian_reservoir_workflow(
            N=gaussian_params["N"],
            J=common_params["J"],
            lambda_reg=common_params["lambda_reg"],
            t_thermalization=common_params["t_thermalization"],
            t_training=common_params["t_training"],
            t_prediction=common_params["t_prediction"],
            sigma=common_params["sigma"],
            rho=common_params["rho"],
            beta=common_params["beta"],
            dt=common_params["dt"],
            input_scale=common_params["input_scale"],
            seed=common_params["seed"],
            alpha=gaussian_params["alpha"],
        )
    )

    return {"human": human_result, "gaussian": gaussian_result}


# ============================================================
# PLOTTING FUNCTIONS
# ============================================================

def plot_comparison_components(Y_pred_h, Y_true_h, Y_pred_g, Y_true_g, dt=0.01, labels=("x", "y", "z")):
    """Plot side-by-side component comparison."""
    t = np.arange(len(Y_pred_h)) * dt
    fig, axes = plt.subplots(3, 2, figsize=(14, 8), sharex=True)

    for i, label in enumerate(labels):
        axes[i, 0].plot(t, Y_true_h[:, i], label=f"true {label}", lw=1.5, color="black")
        axes[i, 0].plot(t, Y_pred_h[:, i], "--", label=f"pred {label}", lw=1.2, color="tomato")
        axes[i, 0].set_ylabel(label)
        axes[i, 0].legend(loc="upper right")
        axes[i, 0].set_title("Human Connectome Reservoir")

        axes[i, 1].plot(t, Y_true_g[:, i], label=f"true {label}", lw=1.5, color="black")
        axes[i, 1].plot(t, Y_pred_g[:, i], "--", label=f"pred {label}", lw=1.2, color="steelblue")
        axes[i, 1].set_ylabel(label)
        axes[i, 1].legend(loc="upper right")
        axes[i, 1].set_title("Gaussian Reservoir")

    axes[-1, 0].set_xlabel("t")
    axes[-1, 1].set_xlabel("t")
    fig.suptitle("Lorenz Components: Human Connectome vs Gaussian Reservoir")
    plt.tight_layout()
    plt.show()


def plot_error_comparison(Y_pred_h, Y_true_h, Y_pred_g, Y_true_g, dt=0.01):
    """Plot error over time for both reservoirs."""
    t = np.arange(len(Y_pred_h)) * dt

    error_h = np.sqrt(((Y_pred_h - Y_true_h) ** 2).sum(axis=1))
    error_g = np.sqrt(((Y_pred_g - Y_true_g) ** 2).sum(axis=1))

    fig, axes = plt.subplots(2, 1, figsize=(12, 6))

    axes[0].plot(t, error_h, label="Human Connectome", lw=1.2, color="tomato")
    axes[0].plot(t, error_g, label="Gaussian", lw=1.2, color="steelblue")
    axes[0].set_xlabel("t")
    axes[0].set_ylabel("L2 Error")
    axes[0].set_title("Prediction Error Over Time")
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    cum_error_h = np.cumsum(error_h) / np.arange(1, len(error_h) + 1)
    cum_error_g = np.cumsum(error_g) / np.arange(1, len(error_g) + 1)

    axes[1].plot(t, cum_error_h, label="Human Connectome", lw=1.5, color="tomato")
    axes[1].plot(t, cum_error_g, label="Gaussian", lw=1.5, color="steelblue")
    axes[1].set_xlabel("t")
    axes[1].set_ylabel("Mean Cumulative Error")
    axes[1].set_title("Cumulative Mean Error")
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    print("\n" + "=" * 60)
    print("ERROR SUMMARY")
    print("=" * 60)
    print(f"Human Connectome - Final error: {error_h[-1]:.4f}, Mean error: {error_h.mean():.4f}")
    print(f"Gaussian         - Final error: {error_g[-1]:.4f}, Mean error: {error_g.mean():.4f}")


def plot_3d_comparison(Y_pred_h, Y_true_h, Y_pred_g, Y_true_g):
    """Plot 3D attractors side by side."""
    fig = plt.figure(figsize=(14, 5))

    ax1 = fig.add_subplot(121, projection="3d")
    ax1.plot(*Y_true_h.T, lw=0.5, color="black", label="true")
    ax1.plot(*Y_pred_h.T, lw=1.2, linestyle="--", color="tomato", label="pred")
    ax1.set_xlabel("x")
    ax1.set_ylabel("y")
    ax1.set_zlabel("z")
    ax1.set_title("Human Connectome Reservoir")
    ax1.legend()

    ax2 = fig.add_subplot(122, projection="3d")
    ax2.plot(*Y_true_g.T, lw=0.5, color="black", label="true")
    ax2.plot(*Y_pred_g.T, lw=1.2, linestyle="--", color="steelblue", label="pred")
    ax2.set_xlabel("x")
    ax2.set_ylabel("y")
    ax2.set_zlabel("z")
    ax2.set_title("Gaussian Reservoir")
    ax2.legend()

    plt.tight_layout()
    plt.show()


In [ ]:
# ============================================================
# CONFIGURATION
# Set shared parameters once, then set model-specific parameters below.
# ============================================================

common_params = {
    "dt": 0.01,
    "sigma": 10.0,
    "rho": 28.0,
    "beta": 8 / 3,
    "J": 0.9,
    "input_scale": 0.5,
    "lambda_reg": 3e-6,
    "seed": 42,
    "t_thermalization": 30.0,
    "t_training": 2000.0,
    "t_prediction": 10.0,
}

human_params = {
    "file_path": NOTEBOOK_DIR / "../data/repeated_10_scale_250/123824_repeated10_scale250.graphml",
    "alpha": 0.3,
}

gaussian_params = {
    "N": 300,
    "alpha": 0.3,
}

print("Shared parameters:")
for key, value in common_params.items():
    print(f"  {key} = {value}")

print("\nHuman connectome parameters:")
for key, value in human_params.items():
    print(f"  {key} = {value}")

print("\nGaussian parameters:")
for key, value in gaussian_params.items():
    print(f"  {key} = {value}")


In [ ]:
# ============================================================
# RUN BOTH RESERVOIRS
# ============================================================

results = run_both_reservoirs(common_params, human_params, gaussian_params)

Y_pred_h = results["human"]["Y_pred"]
Y_true_h = results["human"]["Y_true"]
Y_pred_g = results["gaussian"]["Y_pred"]
Y_true_g = results["gaussian"]["Y_true"]

dt = common_params["dt"]

print("\n" + "=" * 60)
print("RESERVOIR SIZES")
print("=" * 60)
print(f"Human Connectome: {results['human']['N']} nodes")
print(f"Gaussian: {results['gaussian']['N']} nodes")

print("\n" + "=" * 60)
print("RUNTIMES")
print("=" * 60)
print(f"Human Connectome: {results['human']['total_runtime']:.3f} s")
print(f"Gaussian: {results['gaussian']['total_runtime']:.3f} s")


In [ ]:
# ============================================================
# PLOT 1: Component-wise comparison (side by side)
# ============================================================
plot_comparison_components(Y_pred_h, Y_true_h, Y_pred_g, Y_true_g, dt=dt)


In [ ]:
# ============================================================
# PLOT 2: Error over time comparison
# ============================================================
plot_error_comparison(Y_pred_h, Y_true_h, Y_pred_g, Y_true_g, dt=dt)


In [ ]:
# ============================================================
# PLOT 3: 3D attractor comparison
# ============================================================
plot_3d_comparison(Y_pred_h, Y_true_h, Y_pred_g, Y_true_g)
